# Optimize the launch mixture at fixed 81.943 kW

Keep the previous three-mode fiber, Gaussian pulse, and matched linear/nonlinear
simulations. Change only the launch amplitudes:

$$[c_1,c_2,c_3]=[c,100-c,0],\qquad 0\le c\le100.$$

The pulse constructor normalizes these coefficients, so the **total input peak
power stays fixed**. They are amplitudes: the mode-1 power fraction is
$c^2/[c^2+(100-c)^2]$, not $c/100$. Both launched modes have zero relative phase.
All three modes propagate, and mode 3 is allowed to acquire energy.

Minimize the combined mode-1-and-2 distortion:

$$D_{\rm NL}=1-
\frac{\left|\sum_{m=1}^{2}\int A_{{\rm lin},m}^{*}(L,t)
A_{{\rm NL},m}(L,t)\,dt\right|^2}
{\left(\sum_{m=1}^{2}\int |A_{{\rm lin},m}|^2dt\right)
 \left(\sum_{m=1}^{2}\int |A_{{\rm NL},m}|^2dt\right)}.$$

The modal overlaps are added **before** taking the squared magnitude. This keeps
relative phase information and remains defined at both pure-mode endpoints.
The linear reference is recomputed for every candidate launch.

We start at **[50, 50, 0]**. Pure LP01 is a stationary boundary in this setup;
starting with a mixture lets us examine an actual descent path.
The final sweep covers [100, 0, 0] through [0, 100, 0] in steps of 2.
This is a local search in the fixed three-mode, 2048-sample model.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DNL_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DNL_RUN_DIR") ? ENV["DNL_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal amplitude coefficients", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_step_index_81p943kw_power_distri_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitude coefficients", "step_index_81p943kw_power_distri.jl"))

## 1. Settings

Only one coefficient is independent. Use $u=c/100$ internally.
**fd_step = 0.5** controls the coefficient changes used to estimate the gradient.
The optimizer's movement is set by the learning rate and backtracking; it is not
a fixed 0.5 grid search. **sweep_step = 2.0** controls the final demonstration sweep.

The baseline is an equal-amplitude, equal-power launch. Keep at least 50% of its
detected mode-1+2 photons, since normalized DNL alone does not enforce brightness.
Use 5 mm propagation for the search and 1 mm for validation. Run all cells again
after changing the physical settings so that the caches match the new setup.

In [ ]:
fixed_power_W = 81943.10607881194
baseline_coefficient = 50.0
coefficient_bounds = (0.0, 100.0)
relative_phase_rad = 0.0         # mode 2 phase relative to mode 1; fixed here
fd_step = 0.5                    # coefficient units, not propagation distance
sweep_step = 2.0                 # coefficient units
learning_rate = 0.10             # update normalized u = c/100
max_iterations = 20
max_optimizer_evaluations = 80
max_coefficient_change = 10.0
gradient_tolerance = 1e-4         # projected gradient with respect to u
minimum_brightness_fraction = 0.50
search_dz = 0.005                # 5 mm: mixed launches need finer steps
validation_dz = 0.001            # 1 mm
dnl_step_tolerance = 2e-4

settings = Dict(
    "control"=>"amplitude coefficients [c, 100-c, 0]",
    "objective"=>"minimize combined modes 1 and 2 DNL",
    "fixed_power_W"=>fixed_power_W,
    "baseline_coefficient"=>baseline_coefficient,
    "coefficient_bounds"=>collect(coefficient_bounds),
    "relative_phase_rad"=>relative_phase_rad,
    "finite_difference_step_coefficient"=>fd_step, "sweep_step"=>sweep_step,
    "learning_rate_u"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations"=>max_optimizer_evaluations,
    "max_coefficient_change"=>max_coefficient_change,
    "gradient_tolerance_u"=>gradient_tolerance,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_definition"=>"unfiltered mode-1+2 photons relative to equal-launch baseline",
    "search_dz_m"=>search_dz, "validation_dz_m"=>validation_dz,
    "dnl_step_tolerance"=>dnl_step_tolerance,
    "Nt"=>2048, "time_window_ps"=>10.0, "pulse_fwhm_ps"=>0.2,
    "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "objective_modes"=>[1, 2], "objective_filter"=>"none",
    "linear_reference"=>"same candidate input and fiber; n2 = 0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
open(joinpath(run_dir, "settings.toml"), "w") do io
    TOML.print(io, settings)
end
@printf("Fixed total peak power: %.9f kW\n", fixed_power_W/1000)
println("Baseline launch coefficients: ", launch_coefficients(baseline_coefficient))

## 2. Same fiber and three propagated modes

Mode 1 is LP01, mode 2 is LP11-cos, and mode 3 is LP11-sin.
The nonlinear and linear fibers differ only in n2; setting n2 = 0 disables the
nonlinear response. The input to both branches is identical for each candidate.

In [ ]:
time_grid = TimeGrid(settings["Nt"], settings["time_window_ps"])
domain = MMGNLSEDomain(degrees_of_freedom(:time, :space), time_grid)
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
parameters_cp = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=properties.n2, omega0=properties.omega0, raman=properties.raman)


parameters_linear = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=0.0, omega0=properties.omega0, raman=properties.raman)

detected_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=[1, 2])
all_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=:all)
@printf("Three propagated modes; launch and objective in modes 1+2. CP error: %.3g\n",
    S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Simulate a launch and calculate its gradient

Each new call to dnl_at(c) constructs [c,100-c,0], solves the linear and nonlinear
propagation, and compares the final complex fields. Cached calls reuse the result.
The uniform time spacing cancels in the normalized overlap.

There are not two independent optimization variables:

$$\frac{dD}{dc} =
\frac{\partial D}{\partial c_1}-\frac{\partial D}{\partial c_2}
\approx\frac{D(c+0.5)-D(c-0.5)}{1.0}.$$

Each D call also changes c2 and normalizes the launch, so both effects enter the
finite difference. For $u=c/100$, $dD/du=100\,dD/dc$.
Near a bound, use a second-order one-sided derivative without evaluating outside
[0,100]. This is numerical gradient descent; it does not use an adjoint.

In [ ]:
evaluation_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
field_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
evaluations = NamedTuple[]

# Check phase-sensitive multimode overlap and both boundary derivative formulas.
let a = zeros(ComplexF64, 4, 3, 1)
    a[1, 1, 1] = 1
    a[1, 2, 1] = 1
    @assert isapprox(field_fidelity(a, 2exp(0.7im).*a), 1.0; atol=1e-12)
    b = copy(a)
    b[:, 2, :] .*= -1
    @assert field_fidelity(a, b) == 0.0  # relative phase matters
    b = copy(a)
    b[:, 3, :] .= 3
    @assert isapprox(field_fidelity(a, b), 1.0; atol=1e-12)
    @assert field_fidelity(a, b; selected_modes=1:3) < 1.0
    for m in (1, 2)
        pure = zero(a)
        pure[1, m, 1] = 1
        @assert field_fidelity(pure, pure) == 1.0  # both endpoints defined
    end
    for c in (0.0, 0.2, 50.0, 99.8, 100.0)
        points, weights = coefficient_stencil(c)
        derivative = sum(weights .* (points .- 37).^2)
        @assert isapprox(derivative, 2(c-37); atol=1e-9)
    end
end
println("Combined-mode overlap and finite-difference formula checks passed.")

## 4. Check the baseline and gradient

Compare 5 mm and 1 mm propagation. Estimate the same gradient with coefficient
steps 0.5 and 0.25; require agreement within 1%. Set the photon floor using the
fine baseline. These checks happen before gradient descent.

In [ ]:
baseline = dnl_at(baseline_coefficient)
baseline_fine = dnl_at(baseline_coefficient; dz=validation_dz)
g_start = launch_gradient(baseline_coefficient/100)
g_start_half = launch_gradient(baseline_coefficient/100; h=fd_step/2)
gradient_relative_change = abs(g_start-g_start_half)/max(abs(g_start_half), 1e-8)
baseline_step_error = abs(baseline.D_NL-baseline_fine.D_NL)
@assert baseline_step_error < dnl_step_tolerance "Reduce search_dz before optimization."
@assert gradient_relative_change < 0.01 "Refine the finite-difference step."
brightness_floor = minimum_brightness_fraction * baseline_fine.photons
@printf("Gradient dD/dc: %.9f; half-step: %.9f (%.3f%% change)\n",
    g_start/100, g_start_half/100, 100gradient_relative_change)
@printf("Gradient used by descent, dD/du = 100*dD/dc: %.7f\n", g_start)
@printf("%.1f mm versus 1 mm baseline DNL difference: %.9f\n",
    1000search_dz, baseline_step_error)

## 5. Projected gradient descent

Propose $u_{\rm new}=\mathrm{clip}(u-\eta\,dD/du,0,1)$.
Limit the coefficient change to 10, then halve it if brightness fails or DNL
does not decrease sufficiently. Only accepted steps enter the history.
Stop at a small projected gradient, a failed line search, or the budget.
The 20-iteration limit is not a claim of convergence.

In [ ]:
optimization = optimize_launch()
println("Stop: ", optimization.status)
@printf("Selected launch: [%.6f, %.6f, 0]; DNL=%.8f\n",
    optimization.best.coefficient_1, optimization.best.coefficient_2, optimization.best.D_NL)

## 6. Validate the selected launch

Re-evaluate the selected launch at 1 mm and 2.5 mm. Test coefficients 0.5 to
either side at 2.5 mm. These are extra simulations, not another optimization.
Check improvement, brightness, step-size stability, and decreasing accepted DNL.

Report the neighbor comparison and refined gradient separately: a limited search
can improve the objective without reaching a local minimum. This notebook does
not establish convergence in mode count or time sampling, or a global optimum.

In [ ]:
best_coefficient = optimization.best.coefficient_1
best_fine = dnl_at(best_coefficient; dz=validation_dz)
best_mid = dnl_at(best_coefficient; dz=search_dz/2)
left_neighbor = dnl_at(max(coefficient_bounds[1], best_coefficient-fd_step); dz=search_dz/2)
right_neighbor = dnl_at(min(coefficient_bounds[2], best_coefficient+fd_step); dz=search_dz/2)
g_final = launch_gradient(best_coefficient/100)
g_final_half = launch_gradient(best_coefficient/100; h=fd_step/2)
g_final_refined = launch_gradient(best_coefficient/100; h=fd_step/2, dz=validation_dz)
projected_gradient_refined = best_coefficient/100 -
    clamp(best_coefficient/100-g_final_refined, 0.0, 1.0)

improvement_DNL = baseline_fine.D_NL-best_fine.D_NL
best_step_error = abs(optimization.best.D_NL-best_fine.D_NL)
mid_step_error = abs(best_mid.D_NL-best_fine.D_NL)
neighbor_check_passed = best_mid.D_NL <= min(left_neighbor.D_NL, right_neighbor.D_NL)
@assert improvement_DNL > 0 "No validated improvement over the baseline."
@assert best_fine.photons >= brightness_floor "Brightness fails on the finer grid."
@assert max(best_step_error, mid_step_error) < dnl_step_tolerance "Reduce search_dz and rerun."
@assert all(diff([r.D_NL for r in optimization.history]) .< 0)
@assert all(0 <= r.coefficient_1 <= 100 &&
    isapprox(r.coefficient_1+r.coefficient_2, 100; atol=1e-12) &&
    r.coefficient_3 == 0 for r in evaluations)

summary = Dict(
    "status"=>optimization.status, "fixed_power_W"=>fixed_power_W,
    "baseline_coefficients"=>[baseline_coefficient, 100-baseline_coefficient, 0.0],
    "selected_coefficients"=>[best_coefficient, 100-best_coefficient, 0.0],
    "baseline_D_NL"=>baseline_fine.D_NL, "best_D_NL"=>best_fine.D_NL,
    "improvement_DNL"=>improvement_DNL,
    "relative_DNL_reduction"=>improvement_DNL/baseline_fine.D_NL,
    "brightness_ratio"=>best_fine.photons/baseline_fine.photons,
    "baseline_photons"=>baseline_fine.photons, "best_photons"=>best_fine.photons,
    "baseline_step_error"=>baseline_step_error, "best_step_error"=>best_step_error,
    "mid_step_error"=>mid_step_error,
    "initial_gradient_relative_change"=>gradient_relative_change,
    "final_gradient_u"=>g_final, "final_half_step_gradient_u"=>g_final_half,
    "final_refined_gradient_u"=>g_final_refined,
    "final_refined_projected_gradient"=>projected_gradient_refined,
    "refined_gradient_tolerance_reached"=>abs(projected_gradient_refined)<gradient_tolerance,
    "tested_neighbor_check_passed"=>neighbor_check_passed,
    "best_mode1_energy_fraction"=>best_fine.mode1_energy_fraction,
    "best_mode2_energy_fraction"=>best_fine.mode2_energy_fraction,
    "best_mode3_energy_fraction"=>best_fine.mode3_energy_fraction,
    "best_D_NL_all_modes"=>best_fine.D_NL_all_modes,
    "best_output_input_energy_ratio"=>best_fine.output_input_energy_ratio,
    "accepted_steps"=>length(optimization.history)-1,
    "new_optimizer_objective_evaluations"=>optimization.new_objective_evaluations,
    "optimization_seconds"=>optimization.seconds, "cp_relative_error"=>S_cp.relative_error,
    "improvement_and_step_checks_passed"=>true,
    "validation_scope"=>"fixed three-mode model; improved DNL, brightness, and propagation step",
)
open(joinpath(run_dir, "summary.toml"), "w") do io
    TOML.print(io, summary)
end
@printf("Validated DNL: %.8f → %.8f (%.2f%% reduction)\n",
    baseline_fine.D_NL, best_fine.D_NL, 100summary["relative_DNL_reduction"])
@printf("Mode-1+2 photons retained: %.2f%%\n", 100summary["brightness_ratio"])
@printf("Selected launch: [%.6f, %.6f, 0]\n", best_coefficient, 100-best_coefficient)
@printf("Input power fractions: %.3f%% / %.3f%% / 0%%\n",
    100best_fine.input_mode1_fraction, 100best_fine.input_mode2_fraction)
println("No worse than the two tested neighbors: ", neighbor_check_passed)
@printf("Refined projected gradient: %.7g; tolerance: %.1g\n",
    abs(projected_gradient_refined), gradient_tolerance)

## 7. Sweep the full range and overlay gradient descent

The first coefficient goes from 100 to 0 in steps of 2; the second is 100 minus
the first. This gives 51 simulated launches. Each point solves both propagation
branches unless cached. It is a diagnostic sweep after optimization: its best
point does not replace the gradient-descent result.

Blue shows the sweep; orange arrows show accepted gradient steps in order.
Black crosses are the separate 1 mm checks. The second panel shows DNL against
iteration. A lower sweep point demonstrates the limitation of a local search.

In [ ]:
scan_coefficients = collect(coefficient_bounds[2]:-sweep_step:coefficient_bounds[1])
scan = [dnl_at(c) for c in scan_coefficients]
save_rows("coefficient_sweep.csv", scan)
feasible_scan = filter(r -> r.photons >= brightness_floor, scan)
sweep_best = feasible_scan[argmin([r.D_NL for r in feasible_scan])]
summary["sweep_points"] = length(scan)
summary["sweep_best_coefficient"] = sweep_best.coefficient_1
summary["sweep_best_D_NL_search_resolution"] = sweep_best.D_NL
summary["sweep_has_lower_point"] = sweep_best.D_NL < optimization.best.D_NL
open(joinpath(run_dir, "summary.toml"), "w") do io
    TOML.print(io, summary)
end

fig, axes = subplots(1, 2; figsize=(12.5, 4.6), constrained_layout=true)
axes[1].plot([r.coefficient_1 for r in scan], [r.D_NL for r in scan],
    "o-"; color="#366b9e", markersize=3.5, label="Sweep: step 2 (5 mm)")
path_c = [r.coefficient_1 for r in optimization.history]
path_d = [r.D_NL for r in optimization.history]
axes[1].plot(path_c, path_d, "s--"; color="#cf7c32",
    markersize=4, label="Accepted gradient descent (5 mm)")
for i in 2:length(path_c)
    axes[1].annotate(""; xy=(path_c[i], path_d[i]),
        xytext=(path_c[i-1], path_d[i-1]),
        arrowprops=Dict("arrowstyle"=>"-|>", "color"=>"#cf7c32", "lw"=>1.4))
end
label_indices = [1]
for i in 2:length(path_c)-1
    if abs(path_c[i]-path_c[last(label_indices)]) >= 8 &&
            abs(path_c[i]-path_c[end]) >= 8
        push!(label_indices, i)
    end
end
push!(label_indices, length(path_c))
for i in label_indices
    axes[1].annotate(string(optimization.history[i].iteration);
        xy=(path_c[i], path_d[i]), xytext=(4, 7), textcoords="offset points",
        fontsize=8, color="#9b4b10")
end
axes[1].scatter([path_c[1]], [path_d[1]]; facecolors="white", edgecolors="#cf7c32",
    marker="o", s=95, zorder=6, label="Start [50, 50, 0]")
axes[1].scatter([path_c[end]], [path_d[end]]; color="#cf7c32",
    marker="*", s=135, zorder=6, label="Selected end")
axes[1].scatter([baseline_coefficient, best_coefficient],
    [baseline_fine.D_NL, best_fine.D_NL]; color="black", marker="x",
    s=65, zorder=7, label="1 mm validation")
infeasible = filter(r -> r.photons < brightness_floor, scan)
if !isempty(infeasible)
    axes[1].scatter([r.coefficient_1 for r in infeasible], [r.D_NL for r in infeasible];
        marker="x", color="gray", label="Below brightness floor")
end
axes[1].set_xlabel("First amplitude coefficient c (second = 100 − c)")
axes[1].set_ylabel("Combined mode-1+2 DNL")
axes[1].set_xlim(-2, 102)
axes[1].set_title(@sprintf("Fixed %.3f kW: sweep and descent", fixed_power_W/1000))
axes[1].legend(; fontsize=7.6, loc="best")
axes[2].plot([r.iteration for r in optimization.history], path_d,
    "o-"; color="#cf7c32", markersize=4)
axes[2].set_xlabel("Accepted iteration")
axes[2].set_ylabel("Combined mode-1+2 DNL")
axes[2].set_xticks(0:(length(path_c)>11 ? 2 : 1):optimization.history[end].iteration)
axes[2].set_title("Stop: "*optimization.status)
for ax in axes
    ax.grid(; alpha=0.2)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
end
savefig(joinpath(run_dir, "dnl_launch_optimization.png"); dpi=180)
savefig(joinpath(run_dir, "dnl_launch_optimization.pdf"))
display(fig)
close(fig)
@printf("Lowest feasible sweep sample: [%.1f, %.1f, 0], DNL=%.8f\n",
    sweep_best.coefficient_1, sweep_best.coefficient_2, sweep_best.D_NL)
println("Sweep contains a lower DNL than the descent result: ", summary["sweep_has_lower_point"])
println("The selected result remains the gradient-descent result.")

## 8. Peek at each mode: input, linear output, and nonlinear output

Show the baseline and selected launch at 1 mm, with one column per propagated
mode. Time graphs show power in kW; spectra show power per frequency bin in W
against wavelength, following the original notebooks. They are not densities
per nm. The full complex arrays are also saved.
An empty mode has a flat zero input/reference curve.

In [ ]:
times_ps = collect(time_axis(time_grid))
wavelengths_nm = collect(wavelength_axis(time_grid, properties.omega0)) .* 1e9
wavelength_indices = findall(isfinite.(wavelengths_nm) .& (wavelengths_nm .> 0))
wavelength_indices = wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
mode_names = ("LP01", "LP11-cos", "LP11-sin")
curve_labels = ("Input", "Linear reference", "Nonlinear output")
curve_colors = ("#777777", "#cf7c32", "#366b9e")
curve_styles = (":", "--", "-")

plot_modal_output("baseline", baseline_fine)
plot_modal_output("selected", best_fine)

## 9. Spatial mode shapes and how much light each carries

The three basis shapes are fixed by the fiber. Optimizing the coefficients changes
their weights. The bar charts compare input, linear output, and nonlinear output
energy fractions for the baseline and selected launch.

In [ ]:
x = properties.modes.x
y = properties.modes.y
modes = properties.modes.fields  # axes: y, x, mode
mode_names = ("LP01", "LP11 — cos orientation", "LP11 — sin orientation")

let fig = figure(; figsize=(18, 6), constrained_layout=true)
    for ii = 1:3
        ax = subplot(1, 3, ii)
        # Normalize amplitude only for displaying the fixed mode shapes.
        profile = real.(modes[:, :, ii])
        profile = profile ./ maximum(abs, profile)
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, profile;
            cmap="seismic", vmin=-1, vmax=1, shading="auto", rasterized=true)
        ax.set_title("Mode $ii: "*mode_names[ii])
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        theta = range(0, 2pi; length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),
            settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555", linestyle="--", linewidth=0.8)
        fig.colorbar(mesh; ax, shrink=0.75, label="Relative field amplitude")
    end
    fig.suptitle("The three spatial modes — dashed line marks the fiber core")
    savefig(joinpath(run_dir, "spatial_modes.png"); dpi=160)
    savefig(joinpath(run_dir, "spatial_modes.pdf"))
    display(fig)
    close(fig)
end
npzwrite(joinpath(run_dir, "spatial_mode_fields.npz"),
    Dict("x_m"=>x, "y_m"=>y, "modes_y_x_m"=>modes))


fig, axes = subplots(1, 2; figsize=(12.5, 4.3), constrained_layout=true, sharey=true)
modal_rows = NamedTuple[]
for (case_index, (label, row)) in enumerate((("Baseline", baseline_fine), ("Selected", best_fine)))
    saved = field_cache[(row.coefficient_1, row.dz)]
    for (j, a) in enumerate((saved.input, saved.linear, saved.nonlinear))
        energy = vec(sum(abs2.(a); dims=(1, 3)))
        fractions = energy./sum(energy)
        axes[case_index].bar(collect(1:3).+(j-2)*0.23, 100fractions;
            width=0.23, color=curve_colors[j], label=curve_labels[j])
        for m in 1:3
            push!(modal_rows, (; case=lowercase(label), branch=curve_labels[j],
                mode=m, energy_fraction=fractions[m]))
        end
    end
    axes[case_index].set_xticks(1:3, ["LP01", "LP11-cos", "LP11-sin"])
    axes[case_index].set_ylim(0, 105)
    axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
    axes[case_index].set_title(@sprintf("%s: [%.3f, %.3f, 0]",
        label, row.coefficient_1, row.coefficient_2))
    axes[case_index].legend(; fontsize=8)
    axes[case_index].grid(; axis="y", alpha=0.2)
end
save_rows("modal_energy_fractions.csv", modal_rows)
savefig(joinpath(run_dir, "baseline_selected_modal_fractions.png"); dpi=180)
savefig(joinpath(run_dir, "baseline_selected_modal_fractions.pdf"))
display(fig)
close(fig)

## 10. Baseline and selected beams

Reconstruct the actual beam by adding all three complex modal fields before
squaring. Each input/linear/nonlinear panel is evaluated at its own total-power
peak, whose time is printed. **Every beam-intensity panel uses the same fixed 0–100 W/μm² color scale,**
shared by the baseline, selected result, power-optimization notebook, and
both launch-optimization notebooks. The same color means the same intensity.

The shared bounds are set by beam_intensity_limits_W_per_um2 below. If a future result exceeds 100 W/μm², increase the limit in all three notebooks together; the check prevents silently clipping a bright peak.

In [ ]:
# Use the same absolute intensity scale in all three DNL notebooks.
# This covers the saved baseline, selected, and individual-mode intensity maps.
beam_intensity_limits_W_per_um2 = (0.0, 100.0)
beam_color_ticks = collect(range(beam_intensity_limits_W_per_um2...; length=6))


beam_cases = [("Baseline", baseline_fine), ("Selected", best_fine)]
beam_maps = [[peak_beam_map(a) for a in (
    field_cache[(row.coefficient_1, row.dz)].input,
    field_cache[(row.coefficient_1, row.dz)].linear,
    field_cache[(row.coefficient_1, row.dz)].nonlinear)] for (_, row) in beam_cases]
observed_max = maximum(maximum(m.intensity) for maps in beam_maps for m in maps)
common_max = beam_intensity_limits_W_per_um2[2]
@assert observed_max <= common_max "Increase the shared intensity limit in all three notebooks."
fig, beam_axes = subplots(2, 3; figsize=(16, 9.3), constrained_layout=true)
meshes = []
for (case_index, (label, row)) in enumerate(beam_cases)
    for j in 1:3
        ax = beam_axes[case_index, j]
        m = beam_maps[case_index][j]
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, m.intensity;
            cmap="magma", vmin=beam_intensity_limits_W_per_um2[1], vmax=common_max, shading="auto", rasterized=true)
        push!(meshes, mesh)
        ax.set_title(@sprintf("%s — %s\n[%.2f, %.2f, 0]; peak %.2f kW at %.3f ps",
            label, curve_labels[j], row.coefficient_1, row.coefficient_2,
            m.peak_power_W/1000, m.peak_time_ps); fontsize=10)
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
    end
    maps = beam_maps[case_index]
    npzwrite(joinpath(run_dir, lowercase(label)*"_beam_maps.npz"), Dict(
        "x_um"=>x.*1e6, "y_um"=>y.*1e6,
        "input_W_per_um2"=>maps[1].intensity,
        "linear_W_per_um2"=>maps[2].intensity,
        "nonlinear_W_per_um2"=>maps[3].intensity,
        "peak_times_ps"=>[m.peak_time_ps for m in maps],
        "peak_powers_W"=>[m.peak_power_W for m in maps],
        "shared_color_min_W_per_um2"=>beam_intensity_limits_W_per_um2[1],
        "shared_color_max_W_per_um2"=>common_max))
end
fig.colorbar(meshes[end]; ax=vec(beam_axes), shrink=0.8, ticks=beam_color_ticks,
    label="Intensity at each pulse's peak (W/μm²)")
fig.suptitle(@sprintf("Baseline and selected beams — fixed %.3f kW total input", fixed_power_W/1000))
savefig(joinpath(run_dir, "baseline_selected_beams.png"); dpi=160)
savefig(joinpath(run_dir, "baseline_selected_beams.pdf"))
display(fig)
close(fig)
println("All reconstructed spatial integrals match the modal peak powers.")

## 11. See the individual output modes for baseline and result

Each panel shows only that mode's nonlinear output intensity at its own peak
time. These are weighted mode pictures: the shapes stay fixed, but their
brightness changes. The six panels use the same fixed 0–100 W/μm² scale as every beam-intensity
image across the three DNL notebooks. These separate modal
intensities do not include the interference visible in the combined beams above.

In [ ]:
modal_maps = NamedTuple[]
for (case_index, (label, row)) in enumerate(beam_cases)
    a = field_cache[(row.coefficient_1, row.dz)].nonlinear
    fractions = [row.mode1_energy_fraction, row.mode2_energy_fraction, row.mode3_energy_fraction]
    for m in 1:3
        power_t = vec(sum(abs2.(a[:, m, :]); dims=2))
        k = argmax(power_t)
        intensity = abs2.(modes[:, :, m]) .* power_t[k] ./ 1e12
        push!(modal_maps, (; case_index, mode=m, label, intensity,
            peak_power_W=power_t[k], time_ps=times_ps[k], energy_fraction=fractions[m]))
    end
end
modal_observed_max = maximum(maximum(m.intensity) for m in modal_maps)
modal_common_max = beam_intensity_limits_W_per_um2[2]
@assert modal_observed_max <= modal_common_max "Increase the shared intensity limit in all three notebooks."
fig, modal_axes = subplots(2, 3; figsize=(16, 9.3), constrained_layout=true)
meshes = []
for m in modal_maps
    ax = modal_axes[m.case_index, m.mode]
    mesh = ax.pcolormesh(x.*1e6, y.*1e6, m.intensity;
        cmap="magma", vmin=beam_intensity_limits_W_per_um2[1], vmax=modal_common_max, shading="auto", rasterized=true)
    push!(meshes, mesh)
    ax.set_title(@sprintf("%s — mode %d (%s)\nenergy %.4g%%; peak %.4g kW at %.3f ps",
        m.label, m.mode, mode_names[m.mode], 100m.energy_fraction,
        m.peak_power_W/1000, m.time_ps); fontsize=10)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")
    ax.set_aspect("equal")
end
fig.colorbar(meshes[end]; ax=vec(modal_axes), shrink=0.8, ticks=beam_color_ticks,
    label="Single-mode peak intensity (W/μm²)")
fig.suptitle("Individual nonlinear output modes: baseline and selected launch")
savefig(joinpath(run_dir, "baseline_selected_output_modes.png"); dpi=160)
savefig(joinpath(run_dir, "baseline_selected_output_modes.pdf"))
display(fig)
close(fig)
@printf("Completed: %.3f kW; %d accepted steps; %d optimizer evaluations; %d sweep points.\n",
    fixed_power_W/1000, length(optimization.history)-1,
    optimization.new_objective_evaluations, length(scan))
@printf("Total new objective evaluations including validation and sweep: %d\n", length(evaluations))
println("Results: ", run_dir)